In [6]:
from huggingface_hub import hf_hub_download
import sqlite3
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

In [2]:
db_path = hf_hub_download(
    repo_id="giordano-dm/moltbook-crawl",
    filename="moltbook_upload.db",
    repo_type="dataset"
)

In [ ]:
conn = sqlite3.connect(db_path)

query = """
SELECT 
    c1.author_name AS source, 
    c2.author_name AS target, 
    COUNT(*) as weight
FROM 
    comments c1
JOIN 
    comments c2 ON c1.parent_id = c2.id
WHERE 
    c1.author_id != c2.author_id  -- 排除自己回复自己的数据，聚焦于人际互动
GROUP BY 
    source, target
"""

df_edges = pd.read_sql_query(query, conn)
conn.close()

In [8]:
G = nx.from_pandas_edgelist(
    df_edges, 
    source='source', 
    target='target', 
    edge_attr='weight', 
    create_using=nx.DiGraph()
)

In [9]:

print(f"total nodes (agents who participated in commenting): {G.number_of_nodes()}")
print(f"total edges (comment routes): {G.number_of_edges()}")

total nodes (agents who participated in commenting): 10840
total edges (comment routes): 56372


In [10]:
degree_centrality = nx.degree_centrality(G)
degree_centrality

{'0x11aAgent': 0.00036903773410831256,
 'Doormat': 0.10360734385090875,
 'MonkeNigga': 0.01586862256665744,
 'Shellby': 0.006273641479841313,
 '0x96': 0.0014761509364332502,
 'ClawMD': 0.00645816034689547,
 'ClawdNation': 9.225943352707814e-05,
 'WarrenBuffer': 0.01845188670541563,
 '0xClaw': 0.0009225943352707814,
 'DJsAgent': 0.0014761509364332502,
 'Duncan': 0.011071132023249377,
 'Starclawd-1': 0.09631884860226958,
 'ravenclaw': 0.0008303349017437032,
 '0xJB': 0.0007380754682166251,
 'CypherTempre': 0.0023987452717040315,
 'DogJarvis': 0.010609834855613986,
 'Just_Eon19': 0.0008303349017437032,
 'KloKirillTH': 0.007288495248639173,
 'Protocol_Zero': 0.007196235815112095,
 'Rou-Dai': 0.0002767783005812344,
 'TheMoltBank': 0.007288495248639173,
 'kilmon': 0.026847495156379738,
 '0xMiles': 0.0008303349017437032,
 'ElChato': 0.0009225943352707814,
 'EnronEnjoyer': 0.025186825352892333,
 'FiverrClawOfficial': 0.11541655134237476,
 'NEIA': 0.036073438509087556,
 'Snowy': 0.00046129716763

In [11]:
top_users = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:5]
print("core agents (top 5 basing on centrality)")
for user, score in top_users:
    print(f"{user}: {score:.4f}")

core agents (top 5 basing on centrality)
FiverrClawOfficial: 0.1154
Doormat: 0.1036
Starclawd-1: 0.0963
KirillBorovkov: 0.0781
TheLordOfTheDance: 0.0705


In [ ]:
plt.figure(figsize=(14, 10))

# 选择一种布局算法 (spring_layout 适合社交网络，它会让联系紧密的节点靠在一起)
pos = nx.spring_layout(G, k=0.3, seed=42)

# 根据“度中心性”动态设置节点大小 (中心性越高的用户，圆圈越大)
node_sizes = [50 + 5000 * degree_centrality[node] for node in G.nodes()]

# 根据“回复次数(权重)”动态设置边的粗细
edge_weights = [G[u][v]['weight'] * 0.8 for u, v in G.edges()]

# 1. 画节点
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='skyblue', alpha=0.9, edgecolors='white')

# 2. 画边 (带箭头)
nx.draw_networkx_edges(G, pos, width=edge_weights, arrowsize=10, alpha=0.4, edge_color='gray', connectionstyle="arc3,rad=0.1")

# 3. 画标签 (只给重要节点打标签，避免图面过于拥挤)
# 设定一个阈值，比如只显示度中心性排名前 10 的用户名
top_10_users = [user for user, score in sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:10]]
labels = {node: node if node in top_10_users else "" for node in G.nodes()}
nx.draw_networkx_labels(G, pos, labels=labels, font_size=9, font_weight='bold')

plt.title("SNA of UI in comments", fontsize=18)
plt.axis('off')  # 隐藏网格和坐标轴
plt.tight_layout()
plt.show()